In [1]:
import pandas as pd
import duckdb
import os

# Get Napoleon File

In [3]:
entities = pd.read_csv('data/entities_filtered_by_languages.csv')

# Napleon Bonaparte 
napoleon_row = entities[entities['wikidata_id'] == 'Q517']

# Make to csv file
napoleon_row.to_csv('data/napoleon_self.csv', index=False)


```
python src/main.py build --input data/napoleon_self.csv --output data/out/napoleon_self.db --languages en
```

In [6]:
con = duckdb.connect('data/out/napoleon_self.db')

rows = con.execute("""
    SELECT wp.language_code, wp.page_title, e.extract
    FROM wiki_page wp 
    JOIN extract e ON e.wiki_page_id = wp.wiki_page_id
    WHERE wp.wikidata_id == 517
""").fetchall()

output_dirs = [
    "data/text_files",
    "../hpc_jobs/hpc_relik/text_napoleon",
    "../hpc_jobs/hpc_relik_large/text_napoleon",
]

for language_code, page_title, extract in rows:
    filename = f"{page_title.replace(' ', '_')}_{language_code}.txt"
    for d in output_dirs:
        with open(os.path.join(d, filename), "w", encoding="utf-8") as f:
            f.write(extract or "")
    print(f"Wrote {filename}")

con.close()

Wrote Napoleon_en.txt


# Check Napoleon I edges

In [8]:
# Get data with old Spotlight predictions
fullmatch_df = pd.read_csv('data/SpotlightWeightSource_0102_0505_fullmatch_corrected.csv')
fullmatch_df.head()

,language_code,source_wikidata_id,target_wikidata_id,weight
0,en,Q6882,Q53003,0.0
1,en,Q6882,Q5912,0.0
2,en,Q6882,Q212719,0.0
3,en,Q6882,Q514998,0.0
4,en,Q6882,Q210134,0.0


In [9]:
napoleon_edges = fullmatch_df[(fullmatch_df['source_wikidata_id'] == 'Q517') & 
                              (fullmatch_df["language_code"]=="en") & 
                              (fullmatch_df["weight"]>0.0)]
len(napoleon_edges)

67

In [ ]:
# Does Anna Pavlovna of Russia have an edge?
napoleon_edges[napoleon_edges['source_wikidata_id'] == 'Q160800']

,language_code,source_wikidata_id,target_wikidata_id,weight
